# Problema de la mochila

El problema de la mochila es uno de los problemas clásicos de la computación. Consiste en llenar una mochila con objetos de distintos pesos y valores para maximizar el valor total sin superar el peso máximo.

Por ejemplo, supongamos que la mochila tiene una capacidad de 100 kg y que podemos elegir entre estos objetos:

| Objeto | Peso (kg) | Valor (€) |
|---|---:|---:|
| Coca-Cola | 1 | 1,5 |
| Cerveza | 2 | 2,5 |
| Agua | 1 | 0,5 |
| Cacahuetes | 1 | 1,0 |
| Patatas | 2 | 1,5 |
| Galletas | 1 | 1,0 |
| Manzanas | 1 | 1,0 |
| Naranjas | 1 | 1,0 |
| Plátanos | 1 | 1,0 |
| Peras | 1 | 1,0 |
| Fresas | 1 | 1,0 |
| Piña | 1 | 1,0 |
| Manzanas | 1 | 1,0 |

¿Qué combinación de objetos podemos llevar para maximizar el valor sin superar el peso máximo?

El catálogo tiene dos entradas de manzanas con el mismo peso y valor. Las conservamos como dos lotes separados, cada uno con hasta 10 unidades.

Un individuo es una tupla con una cantidad de 0 a 10 por cada fila de la tabla. Por ejemplo, `(0, 2, 0, …)` indica que cogemos dos cervezas. El peso y el valor totales se suman multiplicando cada cantidad por el peso y el valor de una unidad.

In [ ]:
from random import Random

PES_MAXIM = 100
MAX_UNITATS = 10
objectes = [
    ('Coca-Cola', 1, 1.5),
    ('Cerveza', 2, 2.5),
    ('Agua', 1, 0.5),
    ('Cacahuetes', 1, 1.0),
    ('Patatas', 2, 1.5),
    ('Galletas', 1, 1.0),
    ('Manzanas', 1, 1.0),
    ('Naranjas', 1, 1.0),
    ('Plátanos', 1, 1.0),
    ('Peras', 1, 1.0),
    ('Fresas', 1, 1.0),
    ('Piña', 1, 1.0),
    ('Manzanas', 1, 1.0),
]

def pes(individu):
    return sum(q * p for q, (_, p, _) in zip(individu, objectes))

def valor(individu):
    return sum(q * v for q, (_, _, v) in zip(individu, objectes))

def puntuacio(individu):
    if pes(individu) > PES_MAXIM:
        return -1
    return valor(individu)

## De una generación a la siguiente
Ordenamos la población según la puntuación. Conservamos el mejor individuo y creamos el resto de descendientes cruzando parejas de la mitad superior. Una mutación cambia la cantidad de un producto. La mochila vacía es el punto de partida factible de la población.

In [ ]:
def creua(a, b, rng):
    tall = rng.randrange(1, len(a))
    return a[:tall] + b[tall:]

def muta(individu, rng):
    i = rng.randrange(len(individu))
    opcions = [q for q in range(MAX_UNITATS + 1) if q != individu[i]]
    quantitat = rng.choice(opcions)
    return individu[:i] + (quantitat,) + individu[i + 1:]

def genetic(rng, generacions=30):
    poblacio = [(0,) * len(objectes)]
    for _ in range(19):
        poblacio.append(tuple(rng.randrange(MAX_UNITATS + 1) for _ in objectes))

    for _ in range(generacions):
        poblacio.sort(key=puntuacio, reverse=True)
        pares = poblacio[:10]
        nova = [poblacio[0]]
        while len(nova) < len(poblacio):
            a, b = rng.sample(pares, 2)
            fill = creua(a, b, rng)
            if rng.random() < 0.25:
                fill = muta(fill, rng)
            nova.append(fill)
        poblacio = nova
    return max(poblacio, key=puntuacio)

## Resultado
Mostramos los productos elegidos, el peso y el valor. Cambia la semilla o el número de generaciones y compara los resultados.

In [ ]:
resultat = genetic(Random(42))
for quantitat, (nom, _, _) in zip(resultat, objectes):
    if quantitat:
        print(quantitat, nom)
print(f"{pes(resultat)} kg · {valor(resultat):.2f} €")

## Comprobación opcional del óptimo
Hay 11 cantidades posibles para cada una de las 13 entradas. Para calcular el óptimo sin enumerar todas las combinaciones, guardamos el mejor valor para cada capacidad de 0 a 100 kg. Recorremos las capacidades de mayor a menor cada vez que añadimos una unidad, de modo que solo se pueda contar una vez.

In [ ]:
millor_valor = [0] * (PES_MAXIM + 1)
for _, p, v in objectes:
    for _ in range(MAX_UNITATS):
        for capacitat in range(PES_MAXIM, p - 1, -1):
            millor_valor[capacitat] = max(
                millor_valor[capacitat], millor_valor[capacitat - p] + v
            )
print(f"{millor_valor[PES_MAXIM]:.2f} €")